# AesFix K2a: DEAR Dataset Downloader & 512px Preprocessor
**Notebook**: `aesfix-dear-512`
**Hardware**: CPU only (accelerator NONE), Internet ON.
**Role**:
1. Retrieve `HF_TOKEN` from Kaggle Secrets (fail loudly if missing).
2. Download train and test markups from `vsevolodpl/DEAR`.
3. Stream all 29,886 images (4,981 scenes x 6 styles) with 6 worker threads.
4. Downscale long side to 512px, JPEG q92.
5. Exponential backoff with jitter on HTTP 429, honoring Retry-After (up to 8 retries).
6. Delete downloaded raw file (resolving symlinks) and check disk space every 500 files.
7. Resumable across executions; writes `manifest.csv` and verifies <1% missing threshold.


In [ ]:
# Cell 1: Environment & Secrets Setup (FAIL if HF_TOKEN is missing)
import os
import sys
import time
import shutil
import random
import hashlib
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm

start_time = time.time()

# Verify HF_TOKEN from Kaggle Secrets (or attached dataset mount)
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    val = user_secrets.get_secret("HF_TOKEN")
    if val and str(val).strip():
        hf_token = str(val).strip()
        print("[Secrets] HF_TOKEN retrieved and verified successfully from Kaggle Secrets.")
except Exception as e:
    print(f"[Secrets] Note on Kaggle Secrets: {e}")

if not hf_token:
    token_p = Path("/kaggle/input/aesfix-hf-token/token.txt")
    if token_p.exists():
        hf_token = token_p.read_text().strip()
        print("[Auth] HF_TOKEN retrieved from private dataset mount.")

if not hf_token or not str(hf_token).strip():
    print("FATAL: HF_TOKEN missing from Kaggle Secrets and dataset mount!", file=sys.stderr)
    raise SystemExit(1)

os.environ["HF_TOKEN"] = str(hf_token).strip()
print(f"Python: {sys.version}")



In [ ]:
# Cell 2: Dependencies Installation
!pip install --no-cache-dir huggingface_hub requests pillow pandas tqdm



In [ ]:
# Cell 3: Fetch Markups & Define Dataset Partitions
from huggingface_hub import hf_hub_download

OUT_BASE = Path("/kaggle/working")
IMG_DIR = OUT_BASE / "images_512"
IMG_DIR.mkdir(parents=True, exist_ok=True)

# Resume source if previously mounted
INPUT_PREV = Path("/kaggle/input/aesfix-dear-512")

print("Downloading official train and test markups from vsevolodpl/DEAR...")
train_raw_path = hf_hub_download(
    repo_id="vsevolodpl/DEAR",
    filename="train_processed_markup.csv",
    repo_type="dataset",
    token=os.environ["HF_TOKEN"]
)
test_raw_path = hf_hub_download(
    repo_id="vsevolodpl/DEAR",
    filename="test_processed_markup.csv",
    repo_type="dataset",
    token=os.environ["HF_TOKEN"]
)

# Copy markups to working directory for K2b consumption
shutil.copy2(train_raw_path, OUT_BASE / "train_processed_markup.csv")
shutil.copy2(test_raw_path, OUT_BASE / "test_processed_markup.csv")

df_train = pd.read_csv(OUT_BASE / "train_processed_markup.csv")
df_test = pd.read_csv(OUT_BASE / "test_processed_markup.csv")

all_scenes = sorted(list(set(df_train['name'].unique()).union(set(df_test['name'].unique()))))
styles = ['original', 'tiff16_a', 'tiff16_b', 'tiff16_c', 'tiff16_d', 'tiff16_e']

print(f"Total Unique Scenes: {len(all_scenes)} (Train: {df_train['name'].nunique()}, Test: {df_test['name'].nunique()})")
print(f"Total Target Images: {len(all_scenes) * len(styles)} (6 styles per scene)")



In [ ]:
# Cell 4: Multi-Threaded Streaming, Downscaling & Disk Space Monitoring
all_tasks = [(style, scene) for scene in all_scenes for style in styles]
print(f"Starting pipeline for {len(all_tasks)} images with 6 worker threads...")

# Pre-create style directories
for s in styles:
    (IMG_DIR / s).mkdir(parents=True, exist_ok=True)

counter_lock = threading.Lock()
processed_counter = 0

def check_disk(count, total):
    if count % 500 == 0 or count == total:
        total_b, used_b, free_b = shutil.disk_usage("/kaggle/working")
        free_gb = free_b / (1024 ** 3)
        print(f"[{count}/{total}] Free disk space in /kaggle/working: {free_gb:.2f} GB")
        if free_gb < 1.0:
            raise RuntimeError(f"CRITICAL: Free disk space dropped below 1.0 GB ({free_gb:.2f} GB free). Aborting.")

def process_image(task):
    global processed_counter
    style, scene = task
    out_file = IMG_DIR / style / scene
    
    # 1. Resumable check: Already in working dir
    if out_file.exists() and out_file.stat().st_size > 0:
        with open(out_file, "rb") as f:
            h = hashlib.sha256(f.read()).hexdigest()
        with counter_lock:
            processed_counter += 1
            check_disk(processed_counter, len(all_tasks))
        return {"style": style, "scene": scene, "ok": True, "error": "", "bytes": out_file.stat().st_size, "sha256": h}

    # 2. Resumable check: Available in previously mounted input
    prev_file = INPUT_PREV / "images_512" / style / scene
    if prev_file.exists() and prev_file.stat().st_size > 0:
        shutil.copy2(prev_file, out_file)
        with open(out_file, "rb") as f:
            h = hashlib.sha256(f.read()).hexdigest()
        with counter_lock:
            processed_counter += 1
            check_disk(processed_counter, len(all_tasks))
        return {"style": style, "scene": scene, "ok": True, "error": "", "bytes": out_file.stat().st_size, "sha256": h}

    # 3. Download from HF Hub with retry & backoff
    hf_rel_path = f"images/{style}/{scene}"
    tmp_worker_dir = Path(f"/kaggle/temp/hf_raw_th_{threading.get_ident()}")
    tmp_worker_dir.mkdir(parents=True, exist_ok=True)
    
    raw_path = None
    last_err = ""
    for attempt in range(8):
        try:
            raw_path = hf_hub_download(
                repo_id="vsevolodpl/DEAR",
                filename=hf_rel_path,
                repo_type="dataset",
                local_dir=str(tmp_worker_dir),
                token=os.environ["HF_TOKEN"]
            )
            break
        except Exception as e:
            last_err = str(e)
            is_429 = "429" in last_err or (hasattr(e, "response") and getattr(e.response, "status_code", None) == 429)
            retry_after = None
            if hasattr(e, "response") and e.response is not None:
                ra = e.response.headers.get("Retry-After")
                if ra:
                    try:
                        retry_after = float(ra)
                    except ValueError:
                        pass
            if attempt < 7:
                if retry_after is not None:
                    sleep_s = retry_after + random.uniform(0.5, 2.0)
                elif is_429:
                    sleep_s = (2 ** attempt) * 1.5 + random.uniform(0.5, 2.0)
                else:
                    sleep_s = (2 ** attempt) * 0.5 + random.uniform(0.1, 1.0)
                time.sleep(sleep_s)
    
    if raw_path is None or not Path(raw_path).exists():
        with counter_lock:
            processed_counter += 1
            check_disk(processed_counter, len(all_tasks))
        return {"style": style, "scene": scene, "ok": False, "error": last_err[:200], "bytes": 0, "sha256": ""}

    try:
        # 4. Downscale long side to 512px, JPEG q92
        with Image.open(raw_path) as im:
            im = im.convert("RGB")
            w, h = im.size
            if max(w, h) > 512:
                scale = 512.0 / max(w, h)
                nw = max(1, int(round(w * scale)))
                nh = max(1, int(round(h * scale)))
                im = im.resize((nw, nh), Image.Resampling.LANCZOS)
            im.save(str(out_file), format="JPEG", quality=92)
            
        # 5. Delete raw downloaded file (resolving symlinks)
        real_p = Path(raw_path).resolve()
        if real_p.exists():
            try:
                real_p.unlink()
            except Exception:
                pass
        if Path(raw_path).exists():
            try:
                Path(raw_path).unlink()
            except Exception:
                pass
                
        # Clean local thread cache
        hf_cache_dir = Path(os.path.expanduser("~/.cache/huggingface"))
        if hf_cache_dir.exists():
            shutil.rmtree(hf_cache_dir, ignore_errors=True)

        with open(out_file, "rb") as f:
            f_bytes = out_file.stat().st_size
            sha = hashlib.sha256(f.read()).hexdigest()

        with counter_lock:
            processed_counter += 1
            check_disk(processed_counter, len(all_tasks))

        return {"style": style, "scene": scene, "ok": True, "error": "", "bytes": f_bytes, "sha256": sha}
    except Exception as e:
        with counter_lock:
            processed_counter += 1
            check_disk(processed_counter, len(all_tasks))
        return {"style": style, "scene": scene, "ok": False, "error": str(e)[:200], "bytes": 0, "sha256": ""}

with ThreadPoolExecutor(max_workers=6) as executor:
    manifest_records = list(tqdm(executor.map(process_image, all_tasks), total=len(all_tasks), desc="Processing DEAR 512px"))



In [ ]:
# Cell 5: Generate manifest.csv & Fail Loudly if >1% Missing
df_manifest = pd.DataFrame(manifest_records)
manifest_path = OUT_BASE / "manifest.csv"
df_manifest.to_csv(manifest_path, index=False)

n_total = len(df_manifest)
n_ok = int(df_manifest['ok'].sum())
n_missing = n_total - n_ok
pct_missing = (n_missing / n_total) * 100.0

total_b, used_b, free_b = shutil.disk_usage("/kaggle/working")
runtime_sec = time.time() - start_time

print("="*70)
print(f"DEAR 512 MANIFEST SUMMARY")
print("="*70)
print(f"Total Images Expected : {n_total}")
print(f"Successfully Processed: {n_ok}")
print(f"Missing / Failed      : {n_missing} ({pct_missing:.2f}%)")
print(f"Manifest Path         : {manifest_path} ({os.path.getsize(manifest_path)} bytes)")
print(f"Working Directory Size: {sum(f.stat().st_size for f in IMG_DIR.glob('*/*')) / (1024**3):.2f} GB")
print(f"Remaining Free Disk   : {free_b / (1024**3):.2f} GB")
print(f"Total Runtime         : {runtime_sec:.1f}s ({runtime_sec/3600.0:.2f} hours)")
print("="*70)

if pct_missing > 1.0:
    raise RuntimeError(f"FATAL: Missing files {pct_missing:.2f}% exceeds 1.0% limit ({n_missing}/{n_total} files failed)!")
print("[SUCCESS] K2a dataset preprocessor finished within integrity tolerances.")

